In [ ]:
# main code

import torch
import torch.nn as nn
import pandas as pd
import torchaudio
import os
import gc
from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import WavLMModel, Wav2Vec2FeatureExtractor
from torch.nn.utils.rnn import pad_sequence
from torch.amp import autocast, GradScaler

from google.colab import runtime # For session shutdown
import shutil
import sys

# --- 0. Memory Management Optimization ---
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# --- 1. Config ---
MASTER_CSV = '/content/drive/MyDrive/BUU_PHD_THESIS/master_mos_dataset_english_only.csv'
LOCAL_WAV_DIR = '/content/wav_files/'
SAVE_DIR = '/content/drive/MyDrive/BUU_PHD_THESIS/WAVLM_TRAINED_MODELS_ENGLISH_ONLY/'
RESUME_PATH = os.path.join(SAVE_DIR, 'resume_checkpoint.pth')
EMERGENCY_PATH = os.path.join(SAVE_DIR, 'emergency_backup.pth')
BEST_PATH = os.path.join(SAVE_DIR, 'best_model.pth')

BATCH_SIZE = 1           # Physical batch
ACCUMULATION_STEPS = 4   # Effective batch = 4
LEARNING_RATE = 1e-5
NUM_EPOCHS = 50
EARLY_STOP_PATIENCE = 7  # Stop if no improvement in 10 epochs

# --- 2. Model (Optimized for Fine-Tuning) ---
class WavLMTransformerMOS(nn.Module):
    def __init__(self):
        super().__init__()
        self.wavlm = WavLMModel.from_pretrained("microsoft/wavlm-large")
        # Unfreeze all for Fine-Tuning
        for param in self.wavlm.parameters():
            param.requires_grad = True
        self.wavlm.gradient_checkpointing_enable()

        encoder_layer = nn.TransformerEncoderLayer(d_model=1024, nhead=8, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.regression = nn.Linear(1024, 1)

    def forward(self, input_values):
        outputs = self.wavlm(input_values).last_hidden_state
        transformed = self.transformer_encoder(outputs)
        pooled = torch.mean(transformed, dim=1)
        return self.regression(pooled)

# --- 3. Dataset (Cleaned) ---
class MOSDataset(Dataset):
    def __init__(self, df):
        self.df = df
        self.processor = Wav2Vec2FeatureExtractor.from_pretrained("microsoft/wavlm-large")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        filename = str(self.df.iloc[idx, 0]).strip()
        wav_path = os.path.join(LOCAL_WAV_DIR, filename)
        label = float(self.df.iloc[idx, 1])

        try:
            audio, sr = torchaudio.load(wav_path)
            if sr != 16000:
                audio = torchaudio.transforms.Resample(sr, 16000)(audio)
            if audio.shape[0] > 1:
                audio = torch.mean(audio, dim=0, keepdim=True)

            # Extract features
            input_values = self.processor(audio.squeeze(), sampling_rate=16000, return_tensors="pt").input_values
            return input_values.squeeze(), torch.tensor([label], dtype=torch.float32)
        except Exception:
            # Fallback for missing/corrupted files
            return self.__getitem__((idx + 1) % len(self.df))

def collate_fn(batch):
    return pad_sequence([item[0] for item in batch], batch_first=True), torch.stack([item[1] for item in batch])

# --- 4. Training Function ---
def train():
  try:
    # Force clear anything remaining in VRAM
    torch.cuda.empty_cache()
    gc.collect()

    os.makedirs(SAVE_DIR, exist_ok=True)
    device = torch.device("cuda")

    # Load splits
    train_csv_path = '/content/drive/MyDrive/BUU_PHD_THESIS/train_split_english_only.csv'
    val_csv_path = '/content/drive/MyDrive/BUU_PHD_THESIS/val_split_english_only.csv'

    if os.path.exists(train_csv_path) and os.path.exists(val_csv_path):
        print("📂 Loading Laptop splits...")
        train_df = pd.read_csv(train_csv_path)
        val_df = pd.read_csv(val_csv_path)
    else:
        print("🎲 Generating new splits...")
        df = pd.read_csv(MASTER_CSV)
        test_df = df[df.iloc[:, 0].str.contains('urgent_24', na=False)]
        train_val_df = df.drop(test_df.index).sample(frac=1, random_state=42)
        split_idx = int(0.85 * len(train_val_df))
        train_df, val_df = train_val_df.iloc[:split_idx], train_val_df.iloc[split_idx:]

    train_loader = DataLoader(MOSDataset(train_df), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
    val_loader = DataLoader(MOSDataset(val_df), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

    # Move model to GPU (This is where the crash happened before)
    print("🧠 Initializing Model on GPU...")
    model = WavLMTransformerMOS().to(device)

    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE)
    criterion = nn.MSELoss()

    # Corrected modern GradScaler syntax
    scaler = GradScaler('cuda')

    # Resume Logic
    start_epoch, start_batch, best_val_loss, history = 0, 0, float('inf'), []
    latest = None
    if os.path.exists(RESUME_PATH): latest = torch.load(RESUME_PATH)
    emergency = torch.load(EMERGENCY_PATH) if os.path.exists(EMERGENCY_PATH) else None

    if emergency and (not latest or emergency['epoch'] > latest['epoch'] or
                     (emergency['epoch'] == latest['epoch'] and emergency['batch_idx'] > latest['batch_idx'])):
        latest = emergency

    if latest:
        print(f"🔄 Resuming from Epoch {latest['epoch']}, Batch {latest.get('batch_idx', 0)}")

        # 1. Always load the model weights (the most important part)
        model.load_state_dict(latest['model_state_dict'])

        # 2. Check if the optimizer state exists
        if 'optimizer_state_dict' in latest:
            print("📈 Loading optimizer momentum (Full Resume)...")
            optimizer.load_state_dict(latest['optimizer_state_dict'])
        else:
            # This is what happens when resuming from your new 1.2GB Emergency backups
            print("⚠️ Optimizer state not found. Starting with a fresh optimizer (Momentum reset).")
            # Pro-tip: Resetting momentum is fine; the model just takes ~50-100 batches to stabilize.

        # 3. Load other metadata
        start_epoch = latest['epoch']
        start_batch = latest.get('batch_idx', 0) + 1
        best_val_loss = latest.get('best_val_loss', float('inf'))
        history = latest.get('history', [])

    # ... after loading history from checkpoint ...

    patience_counter = 0
    if history:
        # 1. Find the best (lowest) validation loss seen so far
        all_val_losses = [h['val_loss'] for h in history]
        current_best_val = min(all_val_losses)

        # 2. Count how many epochs at the END of history are worse than the best
        # We look at the history in reverse order
        for h in reversed(history):
            if h['val_loss'] <= current_best_val:
                # We found the point where the best model was saved
                break
            patience_counter += 1

        print(f"📊 History analyzed: Last improvement was {patience_counter} epochs ago.")
        print(f"📉 Current Best Val Loss: {current_best_val:.4f}")


    print("🚀 Starting Training...")

    # Training
    for epoch in range(start_epoch, NUM_EPOCHS):
        model.train()
        running_loss = torch.tensor(0.0).to(device)
        optimizer.zero_grad(set_to_none=True)

        loop = tqdm(train_loader, desc=f"Training Epoch {epoch}")
        for i, (inputs, labels) in enumerate(loop):
            if epoch == start_epoch and i < start_batch: continue

            inputs, labels = inputs.to(device), labels.to(device)

            # Modern autocast syntax
            with autocast('cuda'):
                loss = criterion(model(inputs), labels)
                loss = loss / ACCUMULATION_STEPS

            scaler.scale(loss).backward()

            if (i + 1) % ACCUMULATION_STEPS == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                running_loss += (loss.detach() * ACCUMULATION_STEPS)

            # Emergency Save every 5000 batches
            if i % 10000 == 0 and i > 0:
                print(f"\n💾 Attempting emergency save at batch {i}...")

                # Create the checkpoint dictionary
                # We REMOVE the optimizer_state_dict to keep it under 1.3GB
                emergency_data = {
                    'epoch': epoch,
                    'batch_idx': i,
                    'model_state_dict': model.state_dict(),
                    'best_val_loss': best_val_loss,
                    'history': history
                }

                # Step 1: Save to local VM (Instant)
                temp_local_path = "/content/temp_emergency.pth"
                torch.save(emergency_data, temp_local_path)

                if i % 10000 == 0:
                  # Step 2: Copy to Drive using shell command (More stable than torch.save)
                  try:
                      import shutil
                      shutil.copy(temp_local_path, EMERGENCY_PATH)
                      print(f"✅ Emergency backup synced to Drive at batch {i}")
                  except Exception as e:
                      print(f"⚠️ Drive sync failed, but local copy is safe: {e}, Continuing with local backup...")

            if i % 100 == 0: loop.set_postfix(loss=(loss.item() * ACCUMULATION_STEPS))

        start_batch = 0
        avg_train = (running_loss / len(train_loader)).item()

        # Validation
        gc.collect(); torch.cuda.empty_cache()
        model.eval()
        total_val_loss = 0.0
        with torch.inference_mode():
            for inputs, labels in tqdm(val_loader, desc="Validating"):
                total_val_loss += criterion(model(inputs.to(device)), labels.to(device)).item()

        avg_val = total_val_loss / len(val_loader)
        print(f"✅ Epoch {epoch} Done | Train: {avg_train:.4f} | Val: {avg_val:.4f}")

        history.append({'epoch': epoch, 'train_loss': avg_train, 'val_loss': avg_val})

        # 3. Save BEST_MODEL if applicable
        if avg_val < best_val_loss:
            print(f"🌟 Val Loss Improved ({best_val_loss:.4f} -> {avg_val:.4f}). Saving Best Model.")
            patience_counter = 0 # Reset counter
            best_val_loss = avg_val
            print("🌟 New Best Model! Saving...")
            local_best = "/content/local_best.pth"
            # For the BEST model, we save ONLY weights to keep it small (~1.2GB)
            torch.save(model.state_dict(), local_best)
            try:
                shutil.copy(local_best, BEST_PATH)
                print("✅ Best model synced to Drive.")
            except Exception as e:
                print(f"⚠️ Best model Drive sync failed: {e}")

        else:
            patience_counter += 1
            print(f"⚠️ No improvement. Patience: {patience_counter}/{EARLY_STOP_PATIENCE}")

        # --- Check Early Stop Trigger ---
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"🛑 EARLY STOPPING TRIGGERED AT EPOCH {epoch}")
            break

        # --- End of Epoch Saving Logic ---
        # 1. Prepare the full state (3.7 GB)
        full_state = {
            'epoch': epoch + 1,
            'batch_idx': 0,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_loss': best_val_loss,
            'history': history
        }

        # 2. Save RESUME_CHECKPOINT (Local then Drive)
        print("💾 Saving full resume checkpoint (3.7GB)...")
        local_resume = "/content/local_resume.pth"
        torch.save(full_state, local_resume)
        import shutil
        try:
            shutil.copy(local_resume, RESUME_PATH)
            print("✅ Resume checkpoint synced to Drive.")
        except Exception as e:
            print(f"⚠️ Resume Drive sync failed: {e}")

    print("🎊 All Epochs Completed!")

  except Exception as e:
        # If any CRITICAL code error happens (CUDA OOM, Script Error)
        print(f"❌ CRITICAL ERROR IN TRAINING: {e}")
        # We don't want to burn units for an idle broken session.

  finally:
        # THIS IS YOUR SAFETY SWITCH
        print("🔌 SHUTTING DOWN RUNTIME TO SAVE COMPUTE UNITS...")
        # Optional: Save a tiny text file to Drive to tell you why it shut down
        try:
            with open(os.path.join(SAVE_DIR, 'session_log.txt'), 'a') as f:
                f.write(f"\nSession closed at {pd.Timestamp.now()} Istanbul Time.")
        except: pass

        # This kills the VM and stops the billing immediately
        runtime.unassign()

if __name__ == "__main__":
    train()

📂 Loading Laptop splits...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

🧠 Initializing Model on GPU...


config.json: 0.00B [00:00, ?B/s]

...

✅ Epoch 12 Done | Train: 0.0252 | Val: 0.2581
⚠️ No improvement. Patience: 6/7
💾 Saving full resume checkpoint (3.7GB)...
✅ Resume checkpoint synced to Drive.


Training Epoch 13:   0%|          | 0/99021 [00:00<?, ?it/s]


💾 Attempting emergency save at batch 10000...
✅ Emergency backup synced to Drive at batch 10000

💾 Attempting emergency save at batch 20000...
✅ Emergency backup synced to Drive at batch 20000

💾 Attempting emergency save at batch 30000...
✅ Emergency backup synced to Drive at batch 30000

💾 Attempting emergency save at batch 40000...
✅ Emergency backup synced to Drive at batch 40000

💾 Attempting emergency save at batch 50000...
✅ Emergency backup synced to Drive at batch 50000

💾 Attempting emergency save at batch 60000...
✅ Emergency backup synced to Drive at batch 60000

💾 Attempting emergency save at batch 70000...
✅ Emergency backup synced to Drive at batch 70000

💾 Attempting emergency save at batch 80000...
✅ Emergency backup synced to Drive at batch 80000

💾 Attempting emergency save at batch 90000...
✅ Emergency backup synced to Drive at batch 90000


Validating:   0%|          | 0/17484 [00:00<?, ?it/s]

✅ Epoch 13 Done | Train: 0.0270 | Val: 0.2609
⚠️ No improvement. Patience: 7/7
🛑 EARLY STOPPING TRIGGERED AT EPOCH 13
🎊 All Epochs Completed!
🔌 SHUTTING DOWN RUNTIME TO SAVE COMPUTE UNITS...
